# Audit Kualitas Company Report LQ45
**Tanggal:** 26 Sep 2026 · **Jenis:** diagnostik offline · **Cakupan:** 45 report Sectors.app v2.

**Pertanyaan:** apakah data report cukup lengkap, bertipe/bernilai masuk akal, punya provenance utuh, dan aman dipakai untuk quality–valuation–risk serta financial fragility? Tidak ada API call dan tidak dibuat skor saham.

Roadmap: sumber/provenance → struktur/missingness → konsistensi/tanggal → sektor/peer → temuan → batas kesimpulan.

## tl;dr — hasil audit snapshot

- **45/45** report dan sidecar berhasil dibaca; status HTTP 200, endpoint/symbol cocok, dan hash snapshot lokal cocok **45/45**.
- Semua 45 report bertanggal harga **25 Sep 2026**, sedangkan snapshot daily proyek berakhir **24 Sep 2026**. Jangan gabungkan sebagai bukti point-in-time per 24 Sep.
- Cakupan summary: forward P/E **37/45**; proyeksi dan growth forecast **38/45**; annual historical financials **45/45**, tetapi field cash, capex, dan gross profit tidak terisi konsisten. Semua seri annual berakhir di FY2025, panjang **3–8 tahun**.
- Perbedaan sektor penting: lima bank tidak punya cash-and-equivalents/capex/gross-profit di field generik ini; jangan menyimpulkan bahwa mereka tidak punya data keuangan.
- **Kesimpulan penggunaan:** laporan cukup sebagai input awal dossier/screener deskriptif, tetapi **belum cukup untuk menyalakan satu formula score/fragility universal**. Definisi/unit, publication-time, quarterly series, dan penanganan metrik sector-specific masih harus dibereskan.
- Nilai intrinsic value nonpositif pada 15 ticker dan nilai payout di luar 0–1 ditandai tinjauan, bukan dinyatakan salah; definisi/unit model perlu diperiksa.

## Context & Methods

Audit mengukur kehadiran dan konsistensi struktur, bukan kebenaran fundamental. Angka ekstrem/negatif hanya menjadi flag tinjauan, bukan otomatis salah. Tidak ada imputasi. Field dan unit mengikuti JSON; semantik yang tidak tersedia tidak ditebak.

### Key assumptions
- Universe dari snapshot lokal; sumber report adalah data/raw/sectors/company_report/{SYMBOL}/company_report.json, bukan cache lama.
- SHA-256 cocok berarti byte lokal cocok dengan sidecar, bukan bahwa isi fundamentalnya benar.
- Daily snapshot berakhir 24 Sep 2026; report yang baru ditarik memuat harga sampai 25 Sep. Ini risiko timestamp, bukan error konten.
- IQR hanya penanda kandidat review, bukan batas ekonomi.

## Data
Deteksi root memakai IDX_EVIDENCE_LAB_ROOT, direktori project lokal, atau dataset Kaggle. Di Colab mount Drive dan set variabel root. Tidak ada path lokal pembuat yang di-hardcode.

In [1]:
from pathlib import Path
import os, json, hashlib, sys
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 90)

def find_root():
    configured = os.environ.get("IDX_EVIDENCE_LAB_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates += [Path.cwd(), *Path.cwd().parents]
    kaggle = Path("/kaggle/input")
    if kaggle.exists(): candidates += list(kaggle.iterdir())
    for p in candidates:
        if (p / "data/raw/sectors/company_report").is_dir() and (p / "data/raw/sectors/lq45-universe.json").is_file():
            return p.resolve()
    raise FileNotFoundError("Data project tak ditemukan. Mount Drive di Colab/set IDX_EVIDENCE_LAB_ROOT atau lampirkan dataset di Kaggle.")

ROOT = find_root()
REPORT_ROOT = ROOT / "data/raw/sectors/company_report"
UNIVERSE_PATH = ROOT / "data/raw/sectors/lq45-universe.json"
print("Root:", ROOT, "| Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)

Root: /Users/rasyaaudreabramantyawijaya/Downloads/Hackathon Sectors | Python: 3.14.2 | pandas: 2.2.2 | numpy: 2.4.1


### 1. Muat universe, report, dan provenance satu per satu
Baca pasangan JSON dan metadata per ticker. Tidak menggabungkan legacy cache.

In [2]:
universe_payload = json.loads(UNIVERSE_PATH.read_text(encoding="utf-8"))
universe_rows = universe_payload.get("results", []) if isinstance(universe_payload, dict) else universe_payload
symbols = list(dict.fromkeys(str(r.get("symbol", "")).removesuffix(".JK") for r in universe_rows))
reports, metadata, prov_rows, load_errors = {}, {}, [], []
for ticker in symbols:
    rp = REPORT_ROOT / ticker / "company_report.json"
    mp = REPORT_ROOT / ticker / "company_report.metadata.json"
    row = {"symbol":ticker, "report_exists":rp.is_file(), "metadata_exists":mp.is_file(),
           "json_parse_ok":False, "symbol_match":False, "provider":None, "http_status":None,
           "endpoint_match":False, "snapshot_sha256_match":False, "response_sha256_recorded":False, "retrieved_at":None,
           "report_path":str(rp.relative_to(ROOT))}
    try:
        raw = rp.read_bytes()
        doc = json.loads(raw)
        reports[ticker] = doc
        row["json_parse_ok"] = True
        row["symbol_match"] = str(doc.get("symbol","")).removesuffix(".JK") == ticker
        if mp.is_file():
            meta = json.loads(mp.read_text(encoding="utf-8"))
            metadata[ticker] = meta
            row.update(provider=meta.get("provider"), http_status=meta.get("http_status"),
                       endpoint_match=str(meta.get("endpoint","")).rstrip("/").endswith("/company/report/"+ticker),
                       snapshot_sha256_match=hashlib.sha256(raw).hexdigest() == meta.get("snapshot_sha256"),
                       response_sha256_recorded=bool(meta.get("sha256")),
                       retrieved_at=meta.get("retrieved_at"))
    except Exception as e:
        load_errors.append({"symbol":ticker,"error":f"{type(e).__name__}: {e}"})
    prov_rows.append(row)
provenance = pd.DataFrame(prov_rows)
print(f"Universe={len(symbols)} unique tickers; parsed reports={len(reports)}")
display(provenance.head(12))
if load_errors: display(pd.DataFrame(load_errors))

Universe=45 unique tickers; parsed reports=45


,symbol,report_exists,metadata_exists,json_parse_ok,symbol_match,provider,http_status,endpoint_match,snapshot_sha256_match,response_sha256_recorded,retrieved_at,report_path
0,AADI,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T12:13:51.220259+00:00,data/raw/sectors/company_report/AADI/company_report.json
1,ADMR,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:05.155730+00:00,data/raw/sectors/company_report/ADMR/company_report.json
2,ADRO,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:06.226707+00:00,data/raw/sectors/company_report/ADRO/company_report.json
3,AKRA,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:07.074252+00:00,data/raw/sectors/company_report/AKRA/company_report.json
4,AMMN,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:07.878923+00:00,data/raw/sectors/company_report/AMMN/company_report.json
5,AMRT,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:08.770879+00:00,data/raw/sectors/company_report/AMRT/company_report.json
6,ANTM,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:09.647469+00:00,data/raw/sectors/company_report/ANTM/company_report.json
7,ASII,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:10.484787+00:00,data/raw/sectors/company_report/ASII/company_report.json
8,BBCA,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:11.398970+00:00,data/raw/sectors/company_report/BBCA/company_report.json
9,BBNI,True,True,True,True,Sectors.app,200,True,True,True,2026-09-26T15:28:12.273229+00:00,data/raw/sectors/company_report/BBNI/company_report.json


### 2. Struktur, tipe, dan kelengkapan bersarang
Array path dinormalisasi dengan [] ; ticker dihitung terisi bila memiliki minimal satu nilai nonempty. Null, empty, missing, dan angka nol tetap berbeda.

In [3]:
def present(v):
    if v is None: return False
    if isinstance(v,str): return bool(v.strip())
    if isinstance(v,(dict,list)): return bool(v)
    if isinstance(v,float) and np.isnan(v): return False
    return True

def nodes(obj, prefix=""):
    if isinstance(obj,dict):
        for k,v in obj.items():
            p=f"{prefix}.{k}" if prefix else str(k)
            yield p,v
            if isinstance(v,(dict,list)): yield from nodes(v,p)
    elif isinstance(obj,list):
        for v in obj:
            p=prefix+"[]"
            yield p,v
            if isinstance(v,(dict,list)): yield from nodes(v,p)

path_tickers, path_types, path_scalars = defaultdict(set), defaultdict(Counter), defaultdict(list)
for ticker, doc in reports.items():
    found=set()
    for path,v in nodes(doc):
        path_types[path][type(v).__name__]+=1
        if present(v):
            found.add(path)
            if not isinstance(v,(dict,list)): path_scalars[path].append((ticker,v))
    for path in found: path_tickers[path].add(ticker)
field_profile=pd.DataFrame([{"path":p,"tickers_present":len(path_tickers[p]),
 "coverage_pct":100*len(path_tickers[p])/max(len(reports),1),
 "observed_types":", ".join(f"{k}:{n}" for k,n in path_types[p].most_common()),
 "scalar_values":len(path_scalars[p])} for p in sorted(path_types)])
print("Key frequencies across report files:")
display(pd.Series(Counter(k for d in reports.values() for k in d),name="report_count").to_frame().sort_values("report_count",ascending=False))
print("Lowest coverage paths in valuation, financials, dividends, peers:")
rel=field_profile[field_profile.path.str.match(r"^(valuation|financials|dividend|peers)(\.|\[|$)")]
display(rel.sort_values(["coverage_pct","path"]).head(35))

Key frequencies across report files:


,report_count
symbol,45
company_name,45
overview,45
valuation,45
future,45
financials,45
dividend,45
management,45
ownership,45
peers,45


Lowest coverage paths in valuation, financials, dividends, peers:


,path,tickers_present,coverage_pct,observed_types,scalar_values
66,dividend.upcoming_dividends,0,0.0,NoneType:45,0
194,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
196,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
198,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
199,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
200,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
201,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
202,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
203,financials.historical_financials[].industry_breakdown.project_breakdown.PT Bharinto Ek...,0,0.0,NoneType:1,0
206,financials.historical_financials[].industry_breakdown.project_breakdown.PT Graha Panca...,0,0.0,NoneType:1,0


### 3. Kelengkapan metrik inti
Cakupan menyatakan berapa ticker menyediakan data, bukan apakah definisi/unit cocok atau metrik itu valid untuk semua sektor.

In [4]:
def getpath(obj,path):
    vals=[obj]
    for part in path.split("."):
        arr=part.endswith("[]"); key=part[:-2] if arr else part
        nxt=[]
        for x in vals:
            if isinstance(x,dict) and key in x:
                y=x[key]
                if arr and isinstance(y,list): nxt.extend(y)
                elif not arr: nxt.append(y)
        vals=nxt
    return vals

core_paths=[
 "overview.sector","overview.sub_sector","overview.industry","overview.sub_industry","overview.market_cap",
 "overview.latest_close_date","valuation.last_close_price","valuation.latest_close_date","valuation.forward_pe",
 "valuation.intrinsic_value","valuation.historical_valuation[].pe","valuation.historical_valuation[].pb",
 "valuation.historical_valuation[].ps","valuation.historical_valuation[].pcf","valuation.historical_valuation[].peg",
 "valuation.historical_valuation[].pe_peer_avg","future.company_value_forecasts[]","future.company_growth_forecasts[]",
 "financials.eps","financials.historical_eps","financials.historical_financials[].year",
 "financials.historical_financials[].revenue","financials.historical_financials[].earnings",
 "financials.historical_financials[].total_assets","financials.historical_financials[].total_liabilities",
 "financials.historical_financials[].total_equity","financials.historical_financials[].total_debt",
 "financials.historical_financials[].net_debt","financials.historical_financials[].cash_and_equivalents",
 "financials.historical_financials[].operating_cash_flow","financials.historical_financials[].free_cash_flow",
 "financials.historical_financials[].capital_expenditure","financials.historical_financials[].gross_profit",
 "financials.historical_financials[].ebitda","financials.historical_financials[].outstanding_shares",
 "financials.historical_financial_ratio[]","dividend.historical_dividends","dividend.payout_ratio",
 "ownership.major_shareholders[]","peers[].peers_data.companies[]"]
core=[]
for p in core_paths:
    count=0; types=Counter(); total_obs=0; present_obs=0
    annual_match=__import__("re").match(r"financials\.historical_financials\[\]\.(.+)",p)
    for ticker,doc in reports.items():
        vals=getpath(doc,p)
        if any(present(v) for v in vals): count+=1
        if annual_match:
            records=getpath(doc,"financials.historical_financials")
            records=records[0] if records and isinstance(records[0],list) else []
            vals=[row.get(annual_match.group(1)) for row in records if isinstance(row,dict)]
        total_obs+=len(vals)
        present_obs+=sum(present(v) for v in vals)
        types.update(type(v).__name__ for v in vals)
    core.append({"path":p,"tickers_present":count,"expected_tickers":len(reports),
                 "ticker_coverage_pct":round(100*count/max(len(reports),1),1),
                 "nonempty_observations":present_obs,"observation_count":total_obs,
                 "observation_coverage_pct":round(100*present_obs/max(total_obs,1),1),
                 "value_types":", ".join(f"{k}:{n}" for k,n in types.most_common())})
core_coverage=pd.DataFrame(core)
display(core_coverage)

year_rows=[]
for ticker,doc in reports.items():
    arr=getpath(doc,"financials.historical_financials")
    arr=arr[0] if arr and isinstance(arr[0],list) else []
    years=sorted({x.get("year") for x in arr if isinstance(x,dict) and isinstance(x.get("year"),(int,float))})
    year_rows.append({"symbol":ticker,"sector":(doc.get("overview",{}) or {}).get("sector"),
                      "first_year":min(years) if years else None,"last_year":max(years) if years else None,
                      "annual_rows":len(years)})
financial_years=pd.DataFrame(year_rows)
print("Financial annual-history coverage by last fiscal year:")
display(financial_years.groupby(["last_year","sector"],dropna=False).size().rename("tickers").reset_index().sort_values(["last_year","tickers"],ascending=[True,False]))
print("Year-span summary:",financial_years[["first_year","last_year","annual_rows"]].describe().round(1).to_dict())

,path,tickers_present,expected_tickers,ticker_coverage_pct,nonempty_observations,observation_count,observation_coverage_pct,value_types
0,overview.sector,45,45,100.0,45,45,100.0,str:45
1,overview.sub_sector,45,45,100.0,45,45,100.0,str:45
2,overview.industry,45,45,100.0,45,45,100.0,str:45
3,overview.sub_industry,45,45,100.0,45,45,100.0,str:45
4,overview.market_cap,45,45,100.0,45,45,100.0,int:45
5,overview.latest_close_date,45,45,100.0,45,45,100.0,str:45
6,valuation.last_close_price,45,45,100.0,45,45,100.0,int:45
7,valuation.latest_close_date,45,45,100.0,45,45,100.0,str:45
8,valuation.forward_pe,37,45,82.2,37,45,82.2,"float:37, NoneType:8"
9,valuation.intrinsic_value,45,45,100.0,45,45,100.0,int:45


Financial annual-history coverage by last fiscal year:


,last_year,sector,tickers
3,2025,Energy,12
0,2025,Basic Materials,9
2,2025,Consumer Non-Cyclicals,6
4,2025,Financials,5
7,2025,Infrastructures,4
1,2025,Consumer Cyclicals,3
8,2025,Technology,3
6,2025,Industrials,2
5,2025,Healthcare,1


Year-span summary: {'first_year': {'count': 45.0, 'mean': 2019.0, 'std': 0.7, 'min': 2018.0, '25%': 2019.0, '50%': 2019.0, '75%': 2019.0, 'max': 2023.0}, 'last_year': {'count': 45.0, 'mean': 2025.0, 'std': 0.0, 'min': 2025.0, '25%': 2025.0, '50%': 2025.0, '75%': 2025.0, 'max': 2025.0}, 'annual_rows': {'count': 45.0, 'mean': 7.0, 'std': 0.7, 'min': 3.0, '25%': 7.0, '50%': 7.0, '75%': 7.0, 'max': 8.0}}


### 4. Provenance dan keselarasan waktu
SHA-256 report diuji ulang terhadap sidecar. Perbedaan tanggal summary, periode retrieval, dan daily cutoff disajikan terpisah.

In [5]:
checks={"report_file":int(provenance.report_exists.sum()),"metadata_file":int(provenance.metadata_exists.sum()),
 "json_parse":int(provenance.json_parse_ok.sum()),"symbol_match":int(provenance.symbol_match.sum()),
 "provider_sectors":int((provenance.provider=="Sectors.app").sum()),"http_200":int((provenance.http_status==200).sum()),
 "endpoint_match":int(provenance.endpoint_match.sum()),"snapshot_sha256_match":int(provenance.snapshot_sha256_match.sum()),
 "response_sha256_recorded":int(provenance.response_sha256_recorded.sum())}
display(pd.DataFrame([{"check":k,"passed":v,"expected":len(symbols)} for k,v in checks.items()]))
dates=[]
for t,d in reports.items():
    ov=d.get("overview",{}) or {}; va=d.get("valuation",{}) or {}; meta=metadata.get(t,{})
    dates.append({"symbol":t,"sector":ov.get("sector"),"sub_sector":ov.get("sub_sector"),
      "overview_date":ov.get("latest_close_date"),"valuation_date":va.get("latest_close_date"),
      "date_match":ov.get("latest_close_date")==va.get("latest_close_date"),
      "retrieved_at":meta.get("retrieved_at"),"http_status":meta.get("http_status")})
date_audit=pd.DataFrame(dates)
date_audit["retrieved_at_utc"]=pd.to_datetime(date_audit.retrieved_at,utc=True,errors="coerce")
date_audit["close_date"]=pd.to_datetime(date_audit.overview_date,errors="coerce")
date_audit["after_daily_cutoff"]=date_audit.close_date>pd.Timestamp("2026-09-24")
display(date_audit.groupby(["overview_date","valuation_date","after_daily_cutoff"],dropna=False).size().rename("tickers").reset_index())
print("retrieval UTC range:",date_audit.retrieved_at_utc.min(),"to",date_audit.retrieved_at_utc.max())
print("Sector distribution:")
display(date_audit.groupby(["sector","sub_sector"],dropna=False).size().rename("tickers").reset_index().sort_values("tickers",ascending=False))

,check,passed,expected
0,report_file,45,45
1,metadata_file,45,45
2,json_parse,45,45
3,symbol_match,45,45
4,provider_sectors,45,45
5,http_200,45,45
6,endpoint_match,45,45
7,snapshot_sha256_match,45,45
8,response_sha256_recorded,45,45


/var/folders/92/6nhl2gvs7t39bfcqvytg33280000gn/T/ipykernel_17861/3287912652.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  date_audit["retrieved_at_utc"]=pd.to_datetime(date_audit.retrieved_at,utc=True,errors="coerce")
/var/folders/92/

,overview_date,valuation_date,after_daily_cutoff,tickers
0,2026-09-25,2026-09-25,True,45


retrieval UTC range: 2026-09-26 12:13:51.220259+00:00 to 2026-09-26 15:55:25.867881+00:00
Sector distribution:


,sector,sub_sector,tickers
7,Energy,"Oil, Gas & Coal",12
0,Basic Materials,Basic Materials,9
8,Financials,Banks,5
4,Consumer Non-Cyclicals,Food & Beverage,4
12,Infrastructures,Telecommunication,3
14,Technology,Software & IT Services,3
1,Consumer Cyclicals,Apparel & Luxury Goods,1
2,Consumer Cyclicals,Media & Entertainment,1
3,Consumer Cyclicals,Retailing,1
5,Consumer Non-Cyclicals,Food & Staples Retailing,1


### 5. Flag nilai dan struktur untuk ditinjau
Aturan hanya flag. Rasio negatif dapat sah jika denominator negatif; persentase di luar 0–1 bisa disebabkan definisi/unit berbeda.

In [6]:
flags=[]
def addflag(t,kind,evidence,severity="review"): flags.append({"symbol":t,"category":kind,"evidence":str(evidence),"severity":severity})
for t,d in reports.items():
    ov,va=d.get("overview",{}) or {},d.get("valuation",{}) or {}
    if ov.get("latest_close_date")!=va.get("latest_close_date"): addflag(t,"close_date_mismatch",(ov.get("latest_close_date"),va.get("latest_close_date")))
    if ov.get("last_close_price")!=va.get("last_close_price"): addflag(t,"close_price_mismatch",(ov.get("last_close_price"),va.get("last_close_price")))
    if isinstance(ov.get("market_cap"),(int,float)) and ov["market_cap"]<=0: addflag(t,"nonpositive_market_cap",ov["market_cap"])
    if isinstance(ov.get("last_close_price"),(int,float)) and ov["last_close_price"]<=0: addflag(t,"nonpositive_close_price",ov["last_close_price"])
    if isinstance(va.get("forward_pe"),(int,float)) and va["forward_pe"]<=0: addflag(t,"nonpositive_forward_pe",va["forward_pe"],"needs_definition_review")
    if isinstance(va.get("intrinsic_value"),(int,float)) and va["intrinsic_value"]<=0: addflag(t,"nonpositive_intrinsic_value",va["intrinsic_value"],"needs_method_review")
    for arrpath in ["financials.historical_financials","valuation.historical_valuation"]:
        arr=getpath(d,arrpath); arr=arr[0] if arr and isinstance(arr[0],list) else []
        years=[x.get("year") for x in arr if isinstance(x,dict) and x.get("year") is not None]
        if len(years)!=len(set(years)): addflag(t,"duplicate_year:"+arrpath,years)
        if years!=sorted(years): addflag(t,"unordered_years:"+arrpath,years)
    for p in ["dividend.yield_ttm","dividend.dividend_yield_avg","dividend.payout_ratio","dividend.cash_payout_ratio"]:
        for v in getpath(d,p):
            if isinstance(v,(int,float)) and not 0<=v<=1: addflag(t,"ratio_outside_0_1:"+p,v,"needs_unit_definition_review")
    ps=[]
    peer_group_summary=[]
    for group in d.get("peers",[]) or []:
        if not isinstance(group,dict): continue
        pdata=group.get("peers_data",{}) or {}
        companies=pdata.get("companies",[]) if isinstance(pdata,dict) else []
        peer_group_summary.append({"symbol":t,"group_name":pdata.get("group_name"),"peer_rows":len(companies) if isinstance(companies,list) else None,
                                  "own_row_count":sum(1 for x in companies if isinstance(x,dict) and str(x.get("symbol","")).removesuffix(".JK")==t) if isinstance(companies,list) else None})
        for peer in companies if isinstance(companies,list) else []:
            if isinstance(peer,dict) and peer.get("symbol"):
                pt=str(peer["symbol"]).removesuffix(".JK"); ps.append(pt)
                # Company report peer payload intentionally contains a group marked self.
                # Preserve it for auditing, but do not count it as a duplicate or defect.
    dup=[x for x,n in Counter(ps).items() if n>1]
    if dup: addflag(t,"duplicate_peer_symbols",dup)
peer_groups=pd.DataFrame(peer_group_summary)
print("Peer payload shape and self-row are descriptive: API groups intentionally include a self row.")
display(peer_groups.groupby(["peer_rows","own_row_count"],dropna=False).size().rename("ticker_groups").reset_index())
flags=pd.DataFrame(flags,columns=["symbol","category","evidence","severity"])
if len(flags):
    display(flags.groupby(["category","severity"]).size().rename("flag_count").reset_index().sort_values("flag_count",ascending=False))
    display(flags.head(35))
else: print("No flags from these rules.")

Peer payload shape and self-row are descriptive: API groups intentionally include a self row.


,peer_rows,own_row_count,ticker_groups
0,10,1,1


,category,severity,flag_count
2,ratio_outside_0_1:dividend.cash_payout_ratio,needs_unit_definition_review,16
1,nonpositive_intrinsic_value,needs_method_review,15
3,ratio_outside_0_1:dividend.payout_ratio,needs_unit_definition_review,3
0,nonpositive_forward_pe,needs_definition_review,1


,symbol,category,evidence,severity
0,ADMR,nonpositive_intrinsic_value,-987,needs_method_review
1,ADRO,ratio_outside_0_1:dividend.cash_payout_ratio,-0.897356857314066,needs_unit_definition_review
2,AMMN,nonpositive_intrinsic_value,-11850,needs_method_review
3,ANTM,ratio_outside_0_1:dividend.cash_payout_ratio,-2.23697194764983,needs_unit_definition_review
4,ASII,ratio_outside_0_1:dividend.cash_payout_ratio,2.28799423121459,needs_unit_definition_review
5,BBCA,ratio_outside_0_1:dividend.cash_payout_ratio,1.12888987895419,needs_unit_definition_review
6,BBRI,ratio_outside_0_1:dividend.cash_payout_ratio,-3.97467928516739,needs_unit_definition_review
7,BBTN,nonpositive_intrinsic_value,-97,needs_method_review
8,BMRI,ratio_outside_0_1:dividend.cash_payout_ratio,-4.41716516645713,needs_unit_definition_review
9,BRPT,nonpositive_intrinsic_value,-4438,needs_method_review


### 6. Visual kelengkapan field inti
Heatmap biner untuk 45 ticker. Ini matriks missingness, bukan nilai kualitas atau ranking.

In [7]:
heat_paths=["valuation.forward_pe","valuation.intrinsic_value","financials.eps",
 "financials.historical_financials[].revenue","financials.historical_financials[].earnings",
 "financials.historical_financials[].total_assets","financials.historical_financials[].total_equity",
 "financials.historical_financials[].total_debt","financials.historical_financials[].operating_cash_flow",
 "financials.historical_financials[].free_cash_flow","valuation.historical_valuation[].pe_peer_avg",
 "peers[].peers_data.companies[]","dividend.historical_dividends"]
matrix=pd.DataFrame(index=sorted(reports),columns=heat_paths,dtype=float)
for t,d in reports.items():
    for p in heat_paths: matrix.loc[t,p]=float(any(present(v) for v in getpath(d,p)))
fig,ax=plt.subplots(figsize=(14,max(8,len(matrix)*.22)))
im=ax.imshow(matrix.to_numpy(),aspect="auto",interpolation="nearest",cmap="YlGnBu",vmin=0,vmax=1)
labels=[p.replace("financials.historical_financials[].","fin.").replace("valuation.historical_valuation[].","val.").replace("peers[].peers_data.","peer.") for p in heat_paths]
ax.set_yticks(range(len(matrix)),matrix.index,fontsize=8)
ax.set_xticks(range(len(labels)),labels,rotation=50,ha="right",fontsize=8)
ax.set_title(f"Field availability by ticker; n={len(matrix)} (1 present, 0 absent/empty)")
ax.set_xlabel("Selected metric path"); ax.set_ylabel("LQ45 ticker")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j,i,str(int(matrix.iat[i,j])),ha="center",va="center",fontsize=6,color="white" if matrix.iat[i,j] else "black")
fig.colorbar(im,ax=ax,ticks=[0,1],label="Availability (binary)")
fig.tight_layout(); plt.show()
display(pd.DataFrame({"tickers_present":matrix.sum().astype(int),"expected":len(matrix),
 "coverage_pct":(100*matrix.mean()).round(1)}).sort_values("coverage_pct"))

<Figure size 1400x990 with 2 Axes>

,tickers_present,expected,coverage_pct
valuation.forward_pe,37,45,82.2
dividend.historical_dividends,41,45,91.1
valuation.intrinsic_value,45,45,100.0
financials.eps,45,45,100.0
financials.historical_financials[].revenue,45,45,100.0
financials.historical_financials[].earnings,45,45,100.0
financials.historical_financials[].total_assets,45,45,100.0
financials.historical_financials[].total_equity,45,45,100.0
financials.historical_financials[].total_debt,45,45,100.0
financials.historical_financials[].operating_cash_flow,45,45,100.0


## Results & Takeaways
Present tidak berarti benar, comparable, point-in-time, atau cukup untuk production score.

In [8]:
outdir=ROOT/"reports"/"company_report_quality_audit_2026-09-26"
expected_bundle={"audit_summary.json","core_metric_availability_matrix.csv","core_metric_coverage.csv","date_alignment.csv","financial_year_coverage.csv","nested_field_coverage.csv","peer_group_coverage.csv","provenance_checks.csv","review_flags.csv"}
if outdir.exists():
    existing={p.name for p in outdir.iterdir()}
    initial_bundle=expected_bundle-{"financial_year_coverage.csv","peer_group_coverage.csv"}
    if existing not in (initial_bundle,expected_bundle): raise FileExistsError(f"Refusing to overwrite unexpected files in audit bundle: {outdir}")
else:
    outdir.mkdir(parents=True)
provenance.to_csv(outdir/"provenance_checks.csv",index=False)
field_profile.to_csv(outdir/"nested_field_coverage.csv",index=False)
core_coverage.to_csv(outdir/"core_metric_coverage.csv",index=False)
date_audit.drop(columns=["retrieved_at_utc","close_date"]).to_csv(outdir/"date_alignment.csv",index=False)
flags.to_csv(outdir/"review_flags.csv",index=False)
matrix.to_csv(outdir/"core_metric_availability_matrix.csv")
financial_years.to_csv(outdir/"financial_year_coverage.csv",index=False)
peer_groups.to_csv(outdir/"peer_group_coverage.csv",index=False)
summary={"audit_date":"2026-09-26","provider":"Sectors.app","endpoint":"/v2/company/report/{symbol}/",
 "universe_count":len(symbols),"reports_parsed":len(reports),"provenance_checks":checks,
 "close_date_counts":date_audit.overview_date.value_counts(dropna=False).to_dict(),
 "close_after_daily_cutoff":int(date_audit.after_daily_cutoff.sum()),
 "flag_counts":flags.groupby("category").size().to_dict() if len(flags) else {},
 "core_field_coverage":{r.path:int(r.tickers_present) for r in core_coverage.itertuples()},
 "financial_year_last_counts":financial_years.last_year.value_counts(dropna=False).to_dict(),
 "limitations":["Field semantics/units not independently verified","No quarterly-financial endpoint fetched",
 "Outlier and ratio flags need definition review","Current reports are not point-in-time historical snapshots"]}
(outdir/"audit_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print("Audit bundle:",outdir.relative_to(ROOT))
print("Files:",sorted(p.name for p in outdir.iterdir()))
display(pd.Series({"universe":len(symbols),"parsed_reports":len(reports),"snapshot_hash_matches":checks["snapshot_sha256_match"],
 "close_after_daily_cutoff":int(date_audit.after_daily_cutoff.sum()),"review_flags":len(flags)}).to_frame("observed"))
print("Interpretation: structurally complete; not a validated universal score input. Confirm units, sector-specific fields, and metric availability timestamps first.")

Audit bundle: reports/company_report_quality_audit_2026-09-26
Files: ['audit_summary.json', 'core_metric_availability_matrix.csv', 'core_metric_coverage.csv', 'date_alignment.csv', 'financial_year_coverage.csv', 'nested_field_coverage.csv', 'peer_group_coverage.csv', 'provenance_checks.csv', 'review_flags.csv']


,observed
universe,45
parsed_reports,45
snapshot_hash_matches,45
close_after_daily_cutoff,45
review_flags,35


Interpretation: structurally complete; not a validated universal score input. Confirm units, sector-specific fields, and metric availability timestamps first.


### Conclusion, backward trace, and limitations

Work backward: conclusion → executed coverage/flags → path-level null/type checks → local report bytes + timestamp/hash sidecars → LQ45 snapshot → research question about input readiness.

This audit cannot verify field definitions against live schema, audited statements, publication time of nested metrics, point-in-time universe membership, or quarterly completeness. No new API call is implied. A future quarterly-financial fetch needs separate explicit approval. Findings are not investment advice or a stock ranking.